In [67]:
import logging
from datetime import date, datetime
import sys
import os
import numpy as np
import pandas as pd
from pathlib import Path


output_dir = '../data/'
raw_data_dir = '../data/raw_data'
cleaned_data_dir = '../data/cleaned_data'
invalid_data_dir = '../data/invalid_data'
transformed_data_dir = '../data/transformed_data'

log_dir = '../log/'

# ------ Prepare logging ------
log_file = os.path.join(log_dir, f'{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')

LOG_FORMAT = '[%(asctime)s] - %(levelname)s - %(message)s'
DATE_FORMAT = '%Y-%m-%d %H:%M:%S'

# handlers for log file export
file_handler = logging.FileHandler(log_file, encoding='utf-8')
file_handler.setFormatter(logging.Formatter(LOG_FORMAT, DATE_FORMAT))

# handlers for console output
console_handler = logging.StreamHandler(stream=sys.stdout)
console_handler.setFormatter(logging.Formatter(LOG_FORMAT, DATE_FORMAT))

# Attach to root
root = logging.getLogger()
root.setLevel(logging.INFO)
root.handlers.clear()
root.addHandler(file_handler)
root.addHandler(console_handler)

In [ ]:
from module.api_keys import api_keys
from module.data_extraction import download_child_datasets_by_collection_id

# Retreieve relevant API key
api_key = api_keys['data.gov.sg']

# [For improvement] Check if files already exists before starting download
# [For improvement] Update file name as actual file name instead of id

# Here we specify the collection id that we want to download all dataset
# collection_id can be found in https://data.gov.sg/datasets
download_child_datasets_by_collection_id(189, api_key, raw_data_dir)

[2026-03-09 16:17:22] - INFO - Datasets found: 5 in collection_id 189
[2026-03-09 16:17:22] - INFO - 
[2026-03-09 16:17:39] - INFO - Downloaded 10000 rows (total: 10000)
[2026-03-09 16:17:57] - INFO - Downloaded 10000 rows (total: 20000)
[2026-03-09 16:18:16] - INFO - Downloaded 10000 rows (total: 30000)
[2026-03-09 16:18:34] - INFO - Downloaded 10000 rows (total: 40000)
[2026-03-09 16:18:53] - INFO - Downloaded 10000 rows (total: 50000)
[2026-03-09 16:19:12] - INFO - Downloaded 10000 rows (total: 60000)
[2026-03-09 16:19:30] - INFO - Downloaded 10000 rows (total: 70000)
[2026-03-09 16:19:49] - INFO - Downloaded 10000 rows (total: 80000)
[2026-03-09 16:20:08] - INFO - Downloaded 10000 rows (total: 90000)
[2026-03-09 16:20:26] - INFO - Downloaded 10000 rows (total: 100000)
[2026-03-09 16:20:45] - INFO - Downloaded 10000 rows (total: 110000)
[2026-03-09 16:21:04] - INFO - Downloaded 10000 rows (total: 120000)
[2026-03-09 16:21:23] - INFO - Downloaded 10000 rows (total: 130000)
[2026-03-0

In [ ]:
# Read all dataframe from the downloaded raw data

folder = Path(raw_data_dir)

dfs = pd.DataFrame()

dfs = [pd.read_csv(f).drop(columns=["_id"], errors="ignore")
       for f in folder.glob("*.csv")]

# Combine into one DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

In [ ]:
# Make a copy of the dataframe
df = combined_df.copy()

In [55]:
print('**************** datatype ****************')
print(df.info())

print('\n**************** missing values ****************')
print(df.isnull().sum().sort_values(ascending=False))

print('\n**************** overview ****************')
print(df.describe())

**************** datatype ****************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 972674 entries, 0 to 972673
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                972674 non-null  object 
 1   town                 972674 non-null  object 
 2   flat_type            972674 non-null  object 
 3   block                972674 non-null  object 
 4   street_name          972674 non-null  object 
 5   storey_range         972674 non-null  object 
 6   floor_area_sqm       972674 non-null  float64
 7   flat_model           972674 non-null  object 
 8   lease_commence_date  972674 non-null  int64  
 9   resale_price         972674 non-null  float64
 10  remaining_lease      263624 non-null  object 
dtypes: float64(2), int64(1), object(8)
memory usage: 81.6+ MB
None

**************** missing values ****************
remaining_lease        709050
town                        0
month

In [56]:
'''
Observation of dataset
    month 
        * should convert to datetime format to access to datetime function and timeseries analysis
    
    town
        *can consider converting to categorical (i.e. Code 1, 2, 3..etc for each town. Assume there is a master list of town names and codes)
    
    storey_range
        * currently "10 TO 12". Split into two columns - story_range_lower and story_range_upper
     
    resale_price
        * should be integer. May need to discuss with stakeholder if whole value is ok

    floor_area_sqm: 
        * should be INT. Assume there is no decimal value

    remaining_lease:
        * as is "61 years 04 months" is not ideal. Cannot perform mathematiacl operation/ calculation. Cannot be sorted easily. 
        * Better to convert to integer or float (either as 231 months or 5.2 years)
'''

'\nObservation of dataset\n    month \n        * should convert to datetime format to access to datetime function and timeseries analysis\n\n    town\n        *can consider converting to categorical (i.e. Code 1, 2, 3..etc for each town. Assume there is a master list of town names and codes)\n\n    storey_range\n        * currently "10 TO 12". Split into two columns - story_range_lower and story_range_upper\n\n    resale_price\n        * should be integer. May need to discuss with stakeholder if whole value is ok\n\n    floor_area_sqm: \n        * should be INT. Assume there is no decimal value\n\n    remaining_lease:\n        * as is "61 years 04 months" is not ideal. Cannot perform mathematiacl operation/ calculation. Cannot be sorted easily. \n        * Better to convert to integer or float (either as 231 months or 5.2 years)\n'

In [ ]:
import module.data_transform as transform 

# ----- Basic transformation -----

# Correct data column datatype
df['month'] = pd.to_datetime(df['month']) # convert to datetime
df['resale_price'] = df['resale_price'].astype('int') # convert to int
df['floor_area_sqm'] = df['floor_area_sqm'].astype('int') # convert to int

# Split on story 'TO' into lower and upper story
split_df = df['storey_range'].str.split('TO', expand=True)
df['storey_range_lower'] = split_df[0].str.strip().astype(int)
df['storey_range_upper'] = split_df[1].str.strip().astype(int)

# Create remaining_lease_year, and drop the original 'lease_commence_date' column since there are missing values. (lease_commence_date + 99) - month
df['remaining_lease_years'] = np.floor(
    ((pd.to_datetime(df['lease_commence_date'].astype(str) + '-01-01') + pd.DateOffset(years=99) - df['month']).dt.days / 365.25) * 10
) / 10

# drop original remaining_lease col
df.drop(columns=['remaining_lease'], inplace=True)

In [58]:
df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,storey_range_lower,storey_range_upper,remaining_lease_years
0,2012-03-01,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45,Improved,1986,250000,6,10,72.8
1,2012-03-01,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44,Improved,1980,265000,1,5,66.8
2,2012-03-01,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68,New Generation,1980,315000,6,10,66.8
3,2012-03-01,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67,New Generation,1984,320000,1,5,70.8
4,2012-03-01,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67,New Generation,1980,321000,6,10,66.8


In [ ]:
import module.data_validation as validation 

# Define month lower and upper thresholds
lower_bound = '1990-01-01'
upper_bound = datetime.today().strftime("%Y-%m-%d")


# ------- Data Quality check (Pre-transformation) -------

# [For improvement] additional QC for robustness - 
#   check upper/lower story range
#   check block does not exceed 4 alphanumeric
#   check street name belongs to exising address 

# Assumption: a master table exists which stores the values of known towns and addresses
# The table is simulated below as a list
# This also helps to check for any spelling error


known_towns = ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH'
       , 'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI'
       , 'GEYLANG', 'HOUGANG', 'JURONG WEST'
       , 'KALLANG/WHAMPOA', 'MARINE PARADE', 'QUEENSTOWN', 'SENGKANG'
       , 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN'
       , 'LIM CHU KANG', 'SEMBAWANG', 'BUKIT PANJANG', 'PASIR RIS'
       ,'PUNGGOL', 'JURONG EAST'
       ]


# Initialize empty dataframe to collect all invalid records
invalid_records_all = pd.DataFrame()

# Find any unusual month value (should not go lower than 1990 or later than current year-mm)
invalid_month = validation.validate_months(df, lower_bound, upper_bound)
if not invalid_month.empty:
    invalid_records_all = pd.concat([invalid_records_all, invalid_month], ignore_index=True)
    logging.warning(f'Records with invalid months: {len(invalid_month)}')
else: 
    logging.info('QC months passed')

# Find any unusual town value
invalid_towns = validation.validate_town(df, known_towns)
if not invalid_towns.empty:
    invalid_records_all = pd.concat([invalid_records_all, invalid_towns], ignore_index=True)
    logging.warning(f'Records with invalid town - {invalid_towns['town'].unique().tolist()}: {len(invalid_towns)}')
else: 
    logging.info('QC town passed')

# Validate missing values
invalid_missing_values = validation.validate_missing_values(df)
if not invalid_missing_values.empty:
    invalid_records_all = pd.concat([invalid_records_all, invalid_missing_values], ignore_index=True)
    logging.warning(f'Records with missing values: {len(invalid_missing_values)}')
else: 
    logging.info('QC missing values passed')

 # Validate duplicate records
invalid_duplicates = validation.validate_duplicates(df)
if not invalid_duplicates.empty:
    invalid_records_all = pd.concat([invalid_records_all, invalid_duplicates], ignore_index=True)
    logging.warning(f'Records with duplicate records (discard lower resale price): {len(invalid_duplicates)}')
else: 
    logging.info('QC duplicated values passed')   


# Consolidate total invalid records
logging.warning(f'Total invalid records: {len(invalid_records_all)}')

[2026-03-09 21:07:57] - INFO - QC months passed
[2026-03-09 21:07:57] - INFO - QC town passed
[2026-03-09 21:07:57] - INFO - QC missing values passed
[2026-03-09 21:08:01] - WARNING - Records with duplicate records (discard lower resale price): 29283
[2026-03-09 21:08:01] - WARNING - Total invalid records: 29283


In [ ]:
# Drop invalid_record from original df and export for review
# and export as cleaned_dataset for review

cleaned_df = (
    df # original df
    .merge(invalid_records_all, how="left", indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

logging.info(f'Total original records: {len(df)}')
logging.info(f'Total records after cleaning: {len(cleaned_df)}')

cleaned_df.to_csv(rf'{cleaned_data_dir}/cleaned_data_{date.today()}.csv', index=False)
logging.info(f'Cleaned data exported to {cleaned_data_dir}')

[2026-03-09 21:08:06] - INFO - Total original records: 972674
[2026-03-09 21:08:06] - INFO - Total records after cleaning: 941529
[2026-03-09 21:08:09] - INFO - Cleaned data exported to ../data/cleaned_data


In [ ]:
# ------- Using cleaned data, do data transformation -------

# Create resale_identifier column
transform.get_resale_identifier(cleaned_df)

# Create additional helper columns price_per_sqm for QC
cleaned_df['price_per_sqm'] = cleaned_df['resale_price'] / cleaned_df['floor_area_sqm']

In [64]:
cleaned_df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,storey_range_lower,storey_range_upper,remaining_lease_years,invalid_reasons,avg_resale_price,resale_identifier,price_per_sqm
0,2012-03-01,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45,Improved,1986,250000,6,10,72.8,NaN,257500.000000,S1722503A,5555.555556
1,2012-03-01,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44,Improved,1980,265000,1,5,66.8,NaN,257500.000000,S5102503A,6022.727273
2,2012-03-01,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68,New Generation,1980,315000,6,10,66.8,NaN,365877.764706,S6103603A,4632.352941
3,2012-03-01,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67,New Generation,1984,320000,1,5,70.8,NaN,365877.764706,S4743603A,4776.119403
4,2012-03-01,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67,New Generation,1980,321000,6,10,66.8,NaN,365877.764706,S6043603A,4791.044776


In [ ]:
# ------- Data Quality check (Post-transformation) -------

# Initialize empty dataframe to collect all invalid records
invalid_records_post_transform = pd.DataFrame()

# Validate duplicate records
invalid_duplicates = validation.validate_duplicates(cleaned_df)
if not invalid_duplicates.empty:
    invalid_records_post_transform = pd.concat([invalid_records_post_transform, invalid_duplicates], ignore_index=True)
    logging.warning(f'(Post-transform) Records with duplicate records (discard lower resale price): {len(invalid_duplicates)}')
else: 
    logging.info('(Post-transform) QC duplicated values passed')
    
# Consolidate total invalid records
if not invalid_records_post_transform.empty: 
    logging.warning(f'(Post transform) Total invalid records: {len(invalid_records_post_transform)}')
else: 
    logging.info('(Post-transform) No invalid records')

[2026-03-09 21:08:39] - INFO - (Post-transform) QC duplicated values passed
[2026-03-09 21:08:39] - INFO - (Post-transform) No invalid records


In [71]:
# Drop invalid records from post-transformed dataset for review

# Export transformed data
cleaned_df.to_csv(rf'{transformed_data_dir}/transformed_data_{date.today()}.csv', index=False)
logging.info(f'(Post-transformed) Invalid records exported to {invalid_data_dir}invalid_data/invalid_data_post_transform_{date.today()}.csv')

# Export any invalid post-transformed data
if not invalid_records_post_transform.empty:   
    invalid_records_post_transform.to_csv(rf'{invalid_data_dir}/invalid_data_post_transform_{date.today()}.csv', index=False)
    logging.info(f'(Post-transformed) Invalid records exported to {invalid_data_dir}')


[2026-03-09 21:16:36] - INFO - (Post-transformed) Invalid records exported to ../data/invalid_datainvalid_data/invalid_data_post_transform_2026-03-09.csv
